# ARC-AGI-3 Qwen3.6 Duck full harness

This is a controlled return to the public full Duck agent used by the shared TAAF notebook: Qwen3.6 27B FP8 with its original Python tool, prompts, history handling, and solver configuration. It deliberately removes the compact `trajectory` ledger and all associated prompt changes from our previous candidate.

Normal saves use a fast registration path and do not launch Qwen or play the public games. A real competition rerun executes the full agent. Submission plumbing retains fail-fast setup, omits the bundled startup smoke request, and keeps a ten-minute graceful-shutdown buffer; none of those changes alter the agent policy.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False  # noqa: F821
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True  # noqa: F821

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")  # noqa: F821
if not wheelhouse.is_dir():
    raise FileNotFoundError(f"Competition wheelhouse not found at {wheelhouse}.")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(wheelhouse),
        "arc-agi",
    ]
)



In [ ]:
DATASET_SOURCES: list[str] = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]  # noqa: F821
KERNEL_SOURCES: list[str] = []  # noqa: F821
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"  # noqa: F821
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()  # noqa: F821
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0  # noqa: F821
WORKING_DIR.mkdir(parents=True, exist_ok=True)


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# No input audit or smoke-test cell. Exact required paths are asserted by setup.



In [ ]:
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        raise FileNotFoundError(f"TAAF source root not found: {src_root}")
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    if not entries:
        raise RuntimeError(f"TAAF source root contains no importable repositories: {src_root}")
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(_load_setup_env())
    return env


def _remove_bundled_model_smoke_test(command: str) -> str:
    function_start = "\ndef run_vllm_api_smoke_test() -> None:\n"
    function_end = "\n\nprint(f'vLLM wheelhouse path:"
    call = "\nrun_vllm_api_smoke_test()\n"
    if command.count(function_start) != 1 or command.count(function_end) != 1 or command.count(call) != 1:
        raise RuntimeError("The bundled setup command changed; refusing to edit an unrecognized setup script.")
    start = command.index(function_start)
    end = command.index(function_end, start)
    without_function = command[:start] + command[end:]
    return without_function.replace(call, "\n", 1)


def _run_shell_commands(filename: str, *, label: str) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Required {label} command file not found: {path}")
    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list) or not commands:
        raise RuntimeError(f"Required {label} command file is empty or invalid: {path}")
    env = _command_env()
    for command in commands:
        rendered = str(command)
        if filename == "setup_commands.json":
            rendered = _remove_bundled_model_smoke_test(rendered)
        print(f"taaf.kaggle: {label} command", flush=True)
        subprocess.run(rendered, shell=True, check=True, cwd=WORKING_DIR, env=env)
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))

import sysconfig

pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# Start the local Qwen server only for a real competition rerun (or an
# explicitly parameterized full validation). A normal save creates only the
# registration artifact required by Kaggle's code-submission API.
if RUN_AS_SUBMISSION:
    _run_shell_commands("setup_commands.json", label="setup")

    # Setup commands export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
    pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
    for entry in reversed(pythonpath_entries):
        if entry not in sys.path:
            sys.path.insert(0, entry)
else:
    print("Registration mode: Qwen setup and public benchmark are intentionally skipped.")


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime:
    del run_as_submission
    if max_runtime_s <= SOFT_DEADLINE_BUFFER_S:
        raise ValueError(
            f"Deployment runtime {max_runtime_s}s must exceed the {SOFT_DEADLINE_BUFFER_S}s shutdown buffer."
        )
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=max_runtime_s - SOFT_DEADLINE_BUFFER_S)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)



In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
assert float(target.max_runtime_s) == 32400.0
soft_end = _soft_end_time(float(target.max_runtime_s), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR
assert bm.n_passes == 1
assert len(bm.games) == 25
assert bm.solver.concurrency == 28
assert bm.solver.max_runtime_s_per_game == 7920.0



## Original Duck agent unchanged

No tool, prompt, trajectory, memory, or action-policy monkeypatch is applied. The solver loaded from the public TAAF share bundle runs as published.

In [ ]:
if not run_as_submission:
    # Kaggle requires an output file before it can register a notebook version
    # for a code competition. This fast-save artifact is never used or scored
    # during the competition rerun.
    import pandas as pd

    registration = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    registration_path = WORKING_DIR / "submission.parquet"
    registration.to_parquet(registration_path, index=False)
    if not registration_path.is_file():
        raise FileNotFoundError(f"Registration artifact was not written: {registration_path}")
    print(f"Wrote Kaggle code-submission registration artifact: {registration_path}")
else:
    with contextlib.nullcontext():
        preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
        print(preamble)
        print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
        print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
        print(f"deploy.kaggle: competition_rerun       = {true_submission}")
        print(f"deploy.kaggle: soft_end_time           = {soft_end}")
        print("---")

        bundled_git_status = BUNDLE_DIR / "git_status.txt"
        if bundled_git_status.is_file():
            (WORKING_DIR / "git_status.txt").write_text(
                bundled_git_status.read_text(encoding="utf-8"),
                encoding="utf-8",
            )

        if true_submission:
            # Competition reruns use Kaggle's live gateway instead of bundled public games.
            os.environ["ARC_API_KEY"] = "test-key-123"
            os.environ["ARC_BASE_URL"] = "http://gateway:8001/"
            os.environ["SCHEME"] = "http"
            os.environ["HOST"] = "gateway"
            os.environ["PORT"] = "8001"
            os.environ["OPERATION_MODE"] = "competition"
            os.environ["ENVIRONMENTS_DIR"] = ""
            os.environ["RECORDINGS_DIR"] = str(WORKING_DIR / "server_recording")

            deadline = time.monotonic() + 600.0
            last_error = ""
            while time.monotonic() < deadline:
                try:
                    with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                        if response.status < 500:
                            break
                        last_error = f"HTTP {response.status}"
                except Exception as exc:
                    last_error = repr(exc)
                time.sleep(5)
            else:
                raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

            bm.games = _competition_games()
            bm.n_passes = 1
            bm.game_weights = None

        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        _run_shell_commands("teardown_commands.json", label="teardown")
        print("Benchmark and teardown completed successfully.")
